# FireLands: Incorporating the Effect of Wildfires on Sediment Dynamics and Landscape Evolution

*Authors: M. W. de Almeida and B. Campforts*

---

This notebook provides a step-by-step introduction to the **FireLands** Landscape Evolution Model (LEM). FireLands couples the `WildfireGenerator` component with existing Landlab erosion, deposition, and soil production components to simulate how repeated wildfire activity modifies sediment dynamics and landscape form over time.

The workflow is divided into four main stages:

1. Spin up a synthetic landscape to topographic steady state using fluvial incision, hillslope diffusion, and soil production — **no fires yet**.
2. Initialise fuel (vegetation) and run `WildfireGenerator` alone until vegetation reaches a dynamic fire–vegetation equilibrium.
3. Couple `WildfireGenerator` with the erosion and diffusion components via `ErosionChanger` and `DiffusionChanger`, which translate fuel depletion into enhanced erodibility.
4. Visualise the resulting changes in soil thickness and sediment flux at the catchment outlet.

This notebook is a companion to:

> de Almeida M., Shobe C.M., Roda-Boluda D.C., Gourbet L., Veraverbeke S., Distelbrink A., Campforts B. (2026) *FireLands 1.0: A landscape evolution model for simulating the effects of fires and post-fire erosion on sediment dynamics in evolving landscapes.* Geosci Model Dev. *in prep.*

**Prerequisites:** Basic familiarity with Python, NumPy, Matplotlib, and the Landlab modelling toolkit ([Hobley et al., 2017 GMD](https://doi.org/10.5194/gmd-10-4287-2017); [Barnhart et al., 2020 eSurf](https://doi.org/10.5194/esurf-8-379-2020)).

---

## Contents

1. [Import libraries](#1-import-libraries)
2. [Build a steady-state landscape (no fires)](#2-build-a-steady-state-landscape-no-fires)
   - 2.1 [Create the model grid and initial topography](#21-create-the-model-grid-and-initial-topography)
   - 2.2 [Set boundary conditions](#22-set-boundary-conditions)
   - 2.3 [Instantiate erosion, diffusion, and weathering components](#23-instantiate-erosion-diffusion-and-weathering-components)
   - 2.4 [Run to steady state and visualise](#24-run-to-steady-state-and-visualise)
3. [Initialise vegetation and reach fire–vegetation equilibrium](#3-initialise-vegetation-and-reach-firevegation-equilibrium)
   - 3.1 [Instantiate WildfireGenerator and run fire-only loop](#31-instantiate-wildfireGenerator-and-run-fire-only-loop)
   - 3.2 [Visualise vegetation patterns and equilibrium](#32-visualise-vegetation-patterns-and-equilibrium)
4. [Couple fire with erosion and diffusion](#4-couple-fire-with-erosion-and-diffusion)
5. [Post-fire landscape analysis](#5-post-fire-landscape-analysis)

---
## 1. Import libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# FireLands-specific helpers that translate fuel depletion into
# changes in erodibility (ErosionChanger) and diffusivity (DiffusionChanger)
from FireLands import DiffusionChanger, ErosionChanger

from landlab import RasterModelGrid, imshowhs_grid
from landlab.components import (
    DepressionFinderAndRouter,
    DepthDependentTaylorDiffuser,
    ExponentialWeatherer,
    FlowAccumulator,
    SpaceLargeScaleEroder,
    WildfireGenerator,
)

---
## 2. Build a steady-state landscape (no fires)

Before introducing fire, we first evolve the landscape to a topographic steady state where rock uplift is balanced by erosion. This gives `WildfireGenerator` a realistic drainage network and slope field to work with, including river nodes that act as fire breaks.

The steady-state run uses three standard Landlab components:

| Component | Role |
|---|---|
| `SpaceLargeScaleEroder` | Fluvial incision and sediment transport |
| `DepthDependentTaylorDiffuser` | Non-linear hillslope soil creep |
| `ExponentialWeatherer` | Soil production from bedrock |

See the [SpaceLargeScaleEroder tutorial](https://github.com/landlab/landlab/blob/master/docs/source/tutorials/landscape_evolution/space/SPACE_large_scale_eroder_user_guide_and_examples.ipynb) and [DepthDependentTaylorDiffuser tutorial](https://github.com/landlab/landlab/blob/master/docs/source/tutorials/hillslope_geomorphology/depth_dependent_taylor_diffuser/depth_dependent_taylor_diffuser.ipynb) for detailed descriptions of these components.

### 2.1 Create the model grid and initial topography

We use a 50 × 50 `RasterModelGrid` with 10 m node spacing (500 m × 500 m domain). The initial topography is a plane tilted towards the lower-left corner with small random noise to seed a realistic channel network. Both `soil__depth` and `bedrock__elevation` fields are required by the erosion components.

In [ ]:
# --- Grid parameters ---
num_rows = 50
num_columns = 50
node_spacing = 10  # metres

# Node adjacent to the outlet (lower-left corner) used to track sediment flux
node_next_to_outlet = num_columns + 1

# Instantiate the raster grid
mg = RasterModelGrid((num_rows, num_columns), node_spacing)

# --- Topography ---
mg.add_zeros("topographic__elevation", at="node")
np.random.seed(seed=5000)  # fix seed for reproducibility

# Gently tilted plane (towards lower-left) plus small random roughness
topo = mg.node_y / 100000.0 + mg.node_x / 100000.0
random_noise = np.random.rand(len(mg.node_y)) / 1000.0
mg["node"]["topographic__elevation"] += topo + random_noise

# --- Soil and bedrock fields (required by SPACE and DepthDependentTaylorDiffuser) ---
mg.add_zeros("soil__depth", at="node")
mg.at_node["soil__depth"][mg.core_nodes] = 0.0  # no initial soil cover

mg.add_zeros("bedrock__elevation", at="node")
mg.at_node["bedrock__elevation"][:] = mg.at_node["topographic__elevation"]
mg.at_node["topographic__elevation"][:] += mg.at_node["soil__depth"]

### 2.2 Set boundary conditions

All grid edges are closed (no flow exits through the perimeter) except for a single open outlet node at the lower-left corner. This creates a single-outlet watershed, which is the standard configuration for catchment-scale LEMs in Landlab.

In [ ]:
# Close all four grid edges so mass cannot leave through the perimeter
mg.set_closed_boundaries_at_grid_edges(
    bottom_is_closed=True,
    left_is_closed=True,
    right_is_closed=True,
    top_is_closed=True,
)

# Open a single outlet at the lower-left (southwest) corner node (id = 0)
mg.set_watershed_boundary_condition_outlet_id(
    0, mg["node"]["topographic__elevation"], -9999.0
)

### 2.3 Instantiate erosion, diffusion, and weathering components

Key parameter choices for the steady-state spin-up:

| Component | Parameter | Value | Description |
|---|---|---|---|
| `SpaceLargeScaleEroder` | `K_sed` | 0.00015 | Sediment erodibility coefficient |
| `SpaceLargeScaleEroder` | `K_br` | 0.0001 | Bedrock erodibility coefficient |
| `SpaceLargeScaleEroder` | `v_s` | 0.001 | Settling velocity of sediment |
| `ExponentialWeatherer` | `soil_production_maximum_rate` | 0.001 m/yr | Maximum bare-rock weathering rate |
| `ExponentialWeatherer` | `soil_production_decay_depth` | 0.44 m | e-folding depth for weathering |
| `DepthDependentTaylorDiffuser` | `soil_transport_velocity` | 0.001 m/yr | Hillslope diffusivity |
| `DepthDependentTaylorDiffuser` | `slope_crit` | 1.2 | Critical slope for non-linear diffusion |

In [ ]:
# Flow accumulator — required before running any erosion component
fr = FlowAccumulator(mg, flow_director="D8")

# Fluvial incision and sediment transport (SPACE)
sp = SpaceLargeScaleEroder(
    mg,
    K_sed=0.00015,
    K_br=0.0001,
    F_f=0.0,  # no fine fraction
    phi=0.0,  # no sediment porosity
    H_star=1.0,
    v_s=0.001,
    m_sp=0.45,
    n_sp=1.0,
    sp_crit_sed=0,
    sp_crit_br=0,
)

# Soil production from bedrock (exponential decline with soil thickness)
ew = ExponentialWeatherer(
    mg,
    soil_production_maximum_rate=0.001,
    soil_production_decay_depth=0.44,
)

# Non-linear hillslope diffusion
nld = DepthDependentTaylorDiffuser(
    mg,
    soil_transport_velocity=0.001,
    slope_crit=1.2,
    if_unstable="warn",
)

### 2.4 Run to steady state and visualise

The landscape is run for 1 Myr at 200-year timesteps. At each step:

1. Rock uplift is applied to all core bedrock nodes.
2. Flow routing (D8) updates drainage directions and accumulated area.
3. SPACE erodes channels and deposits sediment.
4. Non-linear diffusion redistributes soil on hillslopes.
5. The weatherer converts bedrock to soil.

Sediment flux at the outlet node is stored every step to monitor when the landscape reaches dynamic equilibrium (flat sediment flux curve).

In [ ]:
# --- Clock parameters ---
dt_topo = 200  # timestep (yr)
runtime_topo = 1e6  # total run duration (yr)
ndt_topo = int(runtime_topo / dt_topo)

# Storage for outlet sediment flux
sedflux = np.zeros(ndt_topo)

U = 0.001  # rock uplift rate (m/yr)

for i in range(ndt_topo):
    # 1. Apply uplift to bedrock
    mg.at_node["bedrock__elevation"][mg.core_nodes] += U * dt_topo
    # 2. Route flow and accumulate drainage area
    fr.run_one_step()
    # 3. Fluvial incision and sediment transport
    sp.run_one_step(dt_topo)
    # 4. Hillslope diffusion
    nld.run_one_step(dt_topo)
    # 5. Soil production
    ew.run_one_step()
    # Record outlet flux
    sedflux[i] = mg.at_node["sediment__flux"][node_next_to_outlet]

In [ ]:
# Sediment flux at the outlet through time — a flat curve confirms steady state
plt.figure(figsize=(8, 4))
plt.plot((np.arange(ndt_topo) * dt_topo) / 1000, sedflux, color="black", lw=2)
plt.xlabel("Time (kyr)")
plt.ylabel("Sediment flux at the outlet (m³)")
plt.title("Approach to topographic steady state")
plt.tight_layout()
plt.show()

In [ ]:
# Steady-state topography
imshowhs_grid(
    mg,
    "topographic__elevation",
    ticks_km=False,
    colorbar_label_y=4,
    allow_colorbar=True,
    cbar_or="vertical",
    cbar_height="100%",
    cbar_width="4%",
    add_label_bbox=True,
)
plt.title("Steady-state topographic elevation")
plt.show()

In [ ]:
# Store a snapshot of pre-fire soil depth for later comparison
soils_prefire = mg.at_node["soil__depth"].copy()

# Steady-state soil depth distribution
imshowhs_grid(
    mg,
    "topographic__elevation",
    drape1=mg.at_node["soil__depth"],
    plot_type="Drape1",
    ticks_km=True,
    cmap="pink_r",
    alpha=0.7,
    shrink=0.80,
    colorbar_label_y=4,
    bbox_to_anchor=(0.05, 0, 1, 1),
    vmin=0,
    vmax=1,
    allow_colorbar=True,
    cbar_or="vertical",
    cbar_height="100%",
    cbar_width="4%",
    add_label_bbox=True,
)
plt.title("Soil depth at steady state (pre-fire)")
plt.show()

---
## 3. Initialise vegetation and reach fire–vegetation equilibrium

Before coupling fire to erosion, we first run `WildfireGenerator` in isolation (no erosion) until fuel availability reaches a dynamic equilibrium — the point at which vegetation growth between fires is balanced by fire-driven depletion. This mirrors how real landscapes develop a characteristic fire regime before geomorphic feedbacks become significant.

`WildfireGenerator` requires a `fuel_availability` node field representing the dimensionless fuel (vegetation) load at each node (0 = bare, 1 = fully vegetated). Here we initialise it with a random uniform distribution between 0 and 0.75.

Key parameters used in this example:

| Parameter | Value | Effect |
|---|---|---|
| `potential_fires` | 60 | Mean ignition attempts per year |
| `aridity` | 0.5 | Intermediate climate — balanced fire regime |
| `minimum_river_threshold` | 1×10⁴ m² | Relatively small rivers act as fire breaks |

In [ ]:
# --- Clock parameters for the fire spin-up ---
dt_fire = 1  # timestep (yr) — annual fire cycle
runtime_fire = 200  # total spin-up duration (yr)
ndt_fire = int(runtime_fire / dt_fire)

# Array to track mean fuel availability through the spin-up
fuel_status = np.empty(ndt_fire)

# Create the fuel_availability field (clobber=True overwrites any existing field)
veg = mg.add_zeros("node", "fuel_availability", clobber=True)
veg[:] = np.random.rand(mg.number_of_nodes) * 0.75  # random initial load ∈ [0, 0.75]

### 3.1 Instantiate WildfireGenerator and run fire-only loop

During this stage only `WildfireGenerator` advances — the erosion components are held fixed. At every annual timestep, `run_one_step` fires ignite and spread stochastically across the grid, reduce vegetation at burned nodes by the severity factor, and then vegetation regrows asymptotically toward `max_vegetation` (1.0 by default).

In [ ]:
wg = WildfireGenerator(
    mg,
    potential_fires=60,
    minimum_river_threshold=1e4,
    aridity=0.5,
)

for it in range(ndt_fire):
    wg.run_one_step(dt_fire)
    fuel_status[it] = np.mean(mg.at_node["fuel_availability"])

### 3.2 Visualise vegetation patterns and equilibrium

The spatial map shows where fire activity has left persistent low-fuel patches, and where topographic shielding (e.g. river corridors acting as fire breaks) has allowed vegetation to remain dense. The time series confirms that mean fuel converges to a stable equilibrium value well within the 200-year spin-up window.

In [ ]:
# Spatial distribution of fuel availability after spin-up
imshowhs_grid(
    mg,
    "topographic__elevation",
    drape1=mg.at_node["fuel_availability"],
    plot_type="Drape1",
    ticks_km=True,
    cmap="Greens",
    alpha=0.7,
    shrink=0.80,
    colorbar_label_y=4,
    bbox_to_anchor=(0.05, 0, 1, 1),
    vmin=0,
    vmax=1,
    allow_colorbar=True,
    cbar_or="vertical",
    cbar_height="100%",
    cbar_width="4%",
    add_label_bbox=True,
)
plt.title("Fuel availability at fire–vegetation equilibrium")
plt.show()

In [ ]:
# Mean fuel availability through the spin-up — convergence indicates equilibrium
plt.figure(figsize=(8, 4))
plt.plot(np.arange(ndt_fire) * dt_fire, fuel_status, color="green", lw=2)
plt.xlabel("Year (yr)")
plt.ylabel("Mean fuel availability [-]")
plt.ylim([0.3, 1])
plt.title("Vegetation spin-up: approach to fire–vegetation equilibrium")
plt.tight_layout()
plt.show()

---
## 4. Couple fire with erosion and diffusion

With vegetation at dynamic equilibrium, we now couple `WildfireGenerator` to the erosion and diffusion components. The coupling is mediated by two helper classes from the `FireLands` module:

- **`ErosionChanger`** — reads the `fuel_availability` field each timestep and adjusts the SPACE sediment erodibility coefficient `K_sed`. When fuel is depleted by fire, protective vegetation cover is lost, so `K_sed` increases up to a maximum boost factor.
- **`DiffusionChanger`** — similarly adjusts the `soil_transport_velocity` parameter of `DepthDependentTaylorDiffuser`, increasing hillslope diffusivity when vegetation cover is low.

Both changers implement a `run_one_step` method that must be called after each respective erosion or diffusion step. See de Almeida et al. (2026) for the full parametric relationship between fuel availability and erosion enhancement.

| Parameter | Value | Description |
|---|---|---|
| `K_sed0` | 0.00015 | Background (fully vegetated) sediment erodibility |
| `K_boost` | 100 | Maximum fold-increase in `K_sed` at zero vegetation |
| `diff_0` | 0.001 | Background hillslope diffusivity (m/yr) |
| `diffusion_boost` | 10 | Maximum fold-increase in diffusivity at zero vegetation |

In [ ]:
# --- Clock parameters for the coupled fire–erosion run ---
dt_flands = 1  # timestep (yr) — must match dt_fire for consistency
runtime_flands = 100  # total coupled run duration (yr)
ndt_flands = int(runtime_flands / dt_flands)

# Instantiate the erodibility and diffusivity modifiers
Ec = ErosionChanger(
    mg,
    K_sed0=0.00015,  # background K_sed when vegetation is intact
    K_boost=1e2,  # up to 100× increase at bare soil
)

Dc = DiffusionChanger(
    mg,
    diff_0=0.001,  # background diffusivity
    diffusion_boost=10,  # up to 10× increase at bare soil
)

# Storage for outlet sediment flux during the fire-coupled run
sedflux_fire = np.empty(ndt_flands + 1)

# --- Coupled model loop ---
# Execution order matters: flow routing before erosion;
# ErosionChanger and DiffusionChanger after their respective erosion steps.
for it in range(ndt_flands + 1):
    sedflux_fire[it] = mg.at_node["sediment__flux"][node_next_to_outlet]
    fr.run_one_step()  # update flow directions and drainage area
    wg.run_one_step(dt_flands)  # fire ignition, spread, and vegetation recovery
    ew.run_one_step()  # soil production
    nld.run_one_step(dt_flands)  # hillslope diffusion (with fire-modified diffusivity)
    sp.run_one_step(dt_flands)  # fluvial erosion (with fire-modified K_sed)
    Ec.run_one_step(sp, dt_flands)  # update K_sed based on current fuel_availability
    Dc.run_one_step(
        nld, dt_flands
    )  # update diffusivity based on current fuel_availability

---
## 5. Post-fire landscape analysis

We examine two outputs from the 100-year coupled run:

1. **Soil depth** — compare with the pre-fire snapshot to identify where fires have accelerated erosion and thinned the soil mantle.
2. **Sediment flux at the outlet** — fire-driven erosion pulses should be visible as spikes above the pre-fire baseline.

In [ ]:
# Soil depth after 100 years of coupled fire–erosion activity
imshowhs_grid(
    mg,
    "topographic__elevation",
    drape1=mg.at_node["soil__depth"],
    plot_type="Drape1",
    ticks_km=True,
    cmap="pink_r",
    alpha=0.7,
    shrink=0.80,
    colorbar_label_y=4,
    bbox_to_anchor=(0.05, 0, 1, 1),
    vmin=0,
    vmax=1,
    allow_colorbar=True,
    cbar_or="vertical",
    cbar_height="100%",
    cbar_width="4%",
    add_label_bbox=True,
)
plt.title("Soil depth after 100 years of fire activity")
plt.show()

In [ ]:
# Fire-driven sediment flux at the catchment outlet
# Spikes correspond to years with high-severity or large fires
plt.figure(figsize=(9, 4))
plt.plot(np.arange(ndt_flands + 1), sedflux_fire, color="black", lw=2)
plt.xlabel("Year (yr)")
plt.ylabel("Sediment flux at the outlet (m³)")
plt.title("Fire-driven sediment flux over 100 years")
plt.tight_layout()
plt.show()